In [ ]:
import os
from openai import OpenAI

import math
import matplotlib.pyplot as plt
import re

#hugging face
from huggingface_hub import login
from datasets import load_dataset

#langchain
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage


In [ ]:
#define model "gpt-oss-20b"
MODEL = "openai/gpt-oss-20b:fireworks-ai"
GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"
COLOR_MAP = {"red":RED, "orange": YELLOW, "green": GREEN}
HF_USER = "costadev00"  # Replace with your HuggingFace username  
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)


In [3]:
chat_model = ChatOpenAI(
    api_key=os.environ["HF_TOKEN"],
    base_url="https://router.huggingface.co/v1",
    model=MODEL,
)

chat_response = chat_model.invoke([HumanMessage(content="Hello! How can you assist me today?")])
print(chat_response.content)

Hi there! 👋 I'm here to help with a wide range of topics—from answering questions and brainstorming ideas to providing explanations, suggestions, or even a bit of friendly conversation. How can I assist you today?


In [4]:
import gradio as gr
from langchain_core.messages import AIMessage

def chat(message, history):
    messages = []
    for user, assistant in history:
        messages.append(HumanMessage(content=user))
        messages.append(AIMessage(content=assistant))
    messages.append(HumanMessage(content=message))
    response = chat_model.invoke(messages)
    return response.content

demo = gr.ChatInterface(fn=chat, title="GPT-OSS-20B", description="Chat with openai/gpt-oss-20b")
demo.launch(inline=True)

c:\Users\mathe\Documents\GitHub\llm_engineering\venv\Lib\site-packages\gradio\chat_interface.py:345: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [5]:
# Log in to HuggingFace

hf_token = os.environ["HF_TOKEN"] 
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
DATASET_NAME = f"{HF_USER}/pricer-data"

dataset = load_dataset(DATASET_NAME)
train = dataset['train']
test = dataset['test']

In [7]:
test[2]

{'text': 'How much does this cost to the nearest dollar?\n\nDorman Front Washer Fluid Reservoir Compatible with Select Ford/Lincoln/Mercury Models\nThis washer fluid reservoir is designed to match the fit and function of the original equipment reservoir. It is engineered to withstand the stresses of underhood heat and engine vibration on specified vehicle makes, models, and years. This part is compatible with the following vehicles. Before purchasing, enter your vehicle trim in the garage tool to confirm fitment. Ford Explorer 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010 - Lincoln Aviator 2003, 2004, 2005 - Mercury Mountaineer 2002, 2003, 2004, 2005, \n\nPrice is $',
 'price': 61.68}

In [8]:
def extract_price(s):
    if "$" in s:
      contents = s.split("$")[1]
      contents = contents.replace(',','').replace('$','')
      match = re.search(r"[-+]?\d*\.\d+|\d+", contents)
      return float(match.group()) if match else 0
    return 0

In [9]:
extract_price("The price listed for that Dorman front washer fluid reservoir is approximately **$39** (rounded to the nearest dollar).")

39.0

In [10]:
def model_predict(prompt: str) -> float:
    """Run a single-turn inference and extract the predicted price."""
    response = chat_model.invoke([HumanMessage(content=prompt)])
    content = getattr(response, "content", str(response))
    return extract_price(content)


In [11]:
print(model_predict(test[2]['text']))

20.0


In [12]:
class Tester:

    def __init__(self, predictor, data, title=None, size=250):
        self.predictor = predictor
        self.data = data
        self.title = title or predictor.__name__.replace("_", " ").title()
        self.size = size
        self.guesses = []
        self.truths = []
        self.errors = []
        self.sles = []
        self.colors = []

    def color_for(self, error, truth):
        if error<40 or error/truth < 0.2:
            return "green"
        elif error<80 or error/truth < 0.4:
            return "orange"
        else:
            return "red"

    def run_datapoint(self, i):
        datapoint = self.data[i]
        guess = self.predictor(datapoint["text"])
        truth = datapoint["price"]
        error = abs(guess - truth)
        log_error = math.log(truth+1) - math.log(guess+1)
        sle = log_error ** 2
        color = self.color_for(error, truth)
        title = datapoint["text"].split("\n\n")[1][:20] + "..."
        self.guesses.append(guess)
        self.truths.append(truth)
        self.errors.append(error)
        self.sles.append(sle)
        self.colors.append(color)
        print(f"{COLOR_MAP[color]}{i+1}: Guess: ${guess:,.2f} Truth: ${truth:,.2f} Error: ${error:,.2f} SLE: {sle:,.2f} Item: {title}{RESET}")

    def chart(self, title):
        max_error = max(self.errors)
        plt.figure(figsize=(12, 8))
        max_val = max(max(self.truths), max(self.guesses))
        plt.plot([0, max_val], [0, max_val], color='deepskyblue', lw=2, alpha=0.6)
        plt.scatter(self.truths, self.guesses, s=3, c=self.colors)
        plt.xlabel('Ground Truth')
        plt.ylabel('Model Estimate')
        plt.xlim(0, max_val)
        plt.ylim(0, max_val)
        plt.title(title)
        plt.show()

    def report(self):
        average_error = sum(self.errors) / self.size
        rmsle = math.sqrt(sum(self.sles) / self.size)
        hits = sum(1 for color in self.colors if color=="green")
        title = f"{self.title} Error=${average_error:,.2f} RMSLE={rmsle:,.2f} Hits={hits/self.size*100:.1f}%"
        self.chart(title)

    def run(self):
        self.error = 0
        for i in range(self.size):
            self.run_datapoint(i)
        self.report()

    @classmethod
    def test(cls, function, data):
        cls(function, data).run()

In [13]:
Tester.test(model_predict, test)

1: Guess: $0.00 Truth: $374.41 Error: $374.41 SLE: 35.14 Item: OEM AC Compressor w/...
2: Guess: $0.00 Truth: $225.11 Error: $225.11 SLE: 29.39 Item: Motorcraft YB3125 Fa...
3: Guess: $0.00 Truth: $61.68 Error: $61.68 SLE: 17.12 Item: Dorman Front Washer ...
4: Guess: $349.00 Truth: $599.99 Error: $250.99 SLE: 0.29 Item: HP Premium HD Plus T...
5: Guess: $0.00 Truth: $16.99 Error: $16.99 SLE: 8.35 Item: Super Switch Pickup ...
6: Guess: $0.00 Truth: $31.99 Error: $31.99 SLE: 12.22 Item: Horror Bookmarks, Re...
7: Guess: $78.00 Truth: $101.79 Error: $23.79 SLE: 0.07 Item: SK6241 - Stinger 4 G...
8: Guess: $0.00 Truth: $289.00 Error: $289.00 SLE: 32.15 Item: Godox ML60Bi LED Lig...
9: Guess: $0.00 Truth: $635.86 Error: $635.86 SLE: 41.69 Item: Randall G3 Plus Comb...
10: Guess: $99.99 Truth: $65.99 Error: $34.00 SLE: 0.17 Item: HOLDWILL 6 Pack LED ...
11: Guess: $0.00 Truth: $254.21 Error: $254.21 SLE: 30.71 Item: Viking Horns 3 Gallo...
12: Guess: $0.00 Truth: $412.99 Error: $412.99 SLE

APIStatusError: Error code: 402 - {'error': 'You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.'}